In [1]:
import os
import kagglehub

# Укажите желаемую папку для скачивания
os.environ["KAGGLEHUB_CACHE"] = "/home/fedosdan2/Study/EdgeChess-AI/CV/datasets"  # или "./data" для текущей папки

# Download latest version
path = kagglehub.dataset_download("mmkoya/chessrender360")

print("Path to dataset files:", path)

/home/fedosdan2/miniconda3/envs/myenv/lib/python3.10/site-packages/requests/__init__.py:92: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(
/home/fedosdan2/miniconda3/envs/myenv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


100%|██████████| 24.1G/24.1G [52:52<00:00, 8.15MB/s]  

Extracting files...


Path to dataset files: /home/fedosdan2/Study/EdgeChess-AI/CV/datasets/datasets/mmkoya/chessrender360/versions/5


In [2]:
import os
import json
import shutil
import random
import cv2
from pathlib import Path

# ==========================================
# 1. НАСТРОЙКА ПУТЕЙ И ПАРАМЕТРОВ
# ==========================================
# Укажи путь к папке, где лежит распакованный датасет ChessRender360
DATASET_ROOT = "/home/fedosdan2/Study/EdgeChess-AI/CV/datasets/datasets/mmkoya/chessrender360/versions/5/ChessRender360" 
JSON_DIR = os.path.join(DATASET_ROOT, "annotations") 
IMG_DIR = os.path.join(DATASET_ROOT, "rgb")          

# Куда сохранять готовый YOLO-датасет
OUTPUT_ROOT = "./yolo_chess_dataset_final"
os.makedirs(f"{OUTPUT_ROOT}/images/train", exist_ok=True)
os.makedirs(f"{OUTPUT_ROOT}/images/val", exist_ok=True)
os.makedirs(f"{OUTPUT_ROOT}/labels/train", exist_ok=True)
os.makedirs(f"{OUTPUT_ROOT}/labels/val", exist_ok=True)

# Ресайзим картинки для экономии места и ускорения обучения
TARGET_SIZE = (1000, 1000) 

# Размер маленького bounding box для угла (в пикселях исходного изображения 2000x2000)
CORNER_BBOX_SIZE = 60  

# ==========================================
# 2. КАРТА КЛАССОВ (11 классов)
# ==========================================
CLASS_MAPPING = {
    "pawn": 0,
    "rook": 1,
    "knight": 2,
    "bishop": 3,
    "queen": 4,
    "king": 5,
    "chessboard": 6,
    "corner_white_left": 7,
    "corner_white_right": 8,
    "corner_black_left": 9,
    "corner_black_right": 10
}

# ==========================================
# 3. РАЗДЕЛЕНИЕ ДАННЫХ (80% train / 20% val)
# ==========================================
json_files = [f for f in os.listdir(JSON_DIR) if f.endswith('.json')]
random.seed(42) # Фиксируем seed для воспроизводимости
random.shuffle(json_files)

split_idx = int(len(json_files) * 0.8)
train_files = json_files[:split_idx]
val_files = json_files[split_idx:]

print(f"✅ Найдено {len(json_files)} аннотаций.")
print(f"📊 Train: {len(train_files)} | Val: {len(val_files)}")

# ==========================================
# 4. ФУНКЦИЯ КОНВЕРТАЦИИ ОДНОГО ФАЙЛА
# ==========================================
def convert_to_yolo(json_filename, split_type):
    json_path = os.path.join(JSON_DIR, json_filename)
    
    with open(json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    # Получаем имя файла картинки из JSON
    img_filename = os.path.basename(data.get("rgb_path", ""))
    if not img_filename: 
        return
        
    img_path = os.path.join(IMG_DIR, img_filename)
    if not os.path.exists(img_path): 
        return

    # --- Читаем и ресайзим картинку ---
    img = cv2.imread(img_path)
    if img is None: 
        return
    
    img_resized = cv2.resize(img, TARGET_SIZE, interpolation=cv2.INTER_AREA)
    
    dest_img_path = os.path.join(OUTPUT_ROOT, "images", split_type, img_filename)
    cv2.imwrite(dest_img_path, img_resized)

    new_h, new_w = TARGET_SIZE[1], TARGET_SIZE[0]
    
    # Коэффициенты масштабирования координат
    scale_x = new_w / img.shape[1]
    scale_y = new_h / img.shape[0]

    yolo_lines = []

    # --- 4.1. ДОСКА (frame_bbox) ---
    if "frame_bbox" in data:
        x_min, y_min, x_max, y_max = data["frame_bbox"]
        x_min, x_max = x_min * scale_x, x_max * scale_x
        y_min, y_max = y_min * scale_y, y_max * scale_y
        
        x_center = ((x_min + x_max) / 2.0) / new_w
        y_center = ((y_min + y_max) / 2.0) / new_h
        width = (x_max - x_min) / new_w
        height = (y_max - y_min) / new_h
        
        yolo_lines.append(f"6 {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}")

    # --- 4.2. ФИГУРЫ ---
    if "pieces" in data:
        for piece in data["pieces"]:
            piece_name = piece.get("piece_name", "")
            bbox = piece.get("bbox", [])
            if len(bbox) != 4: continue
            
            # white_pawn -> pawn, black_king -> king
            base_piece = piece_name.split("_")[-1] 
            
            if base_piece in CLASS_MAPPING:
                class_id = CLASS_MAPPING[base_piece]
                x_min, y_min, x_max, y_max = bbox
                
                x_min, x_max = x_min * scale_x, x_max * scale_x
                y_min, y_max = y_min * scale_y, y_max * scale_y
                
                x_center = ((x_min + x_max) / 2.0) / new_w
                y_center = ((y_min + y_max) / 2.0) / new_h
                width = (x_max - x_min) / new_w
                height = (y_max - y_min) / new_h
                
                yolo_lines.append(f"{class_id} {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}")

    # --- 4.3. УГЛЫ ДОСКИ (board_corners) ---
    if "board_corners" in data:
        corners = data["board_corners"]
        corner_mapping = {
            "white_left": "corner_white_left",
            "white_right": "corner_white_right",
            "black_left": "corner_black_left",
            "black_right": "corner_black_right"
        }
        
        for json_key, class_name in corner_mapping.items():
            if json_key in corners:
                px, py = corners[json_key]
                
                # Масштабируем координаты точки
                px_scaled = px * scale_x
                py_scaled = py * scale_y
                
                # Создаем маленький квадрат вокруг точки угла
                half_size = (CORNER_BBOX_SIZE * scale_x) / 2.0
                
                x_min = px_scaled - half_size
                y_min = py_scaled - half_size
                x_max = px_scaled + half_size
                y_max = py_scaled + half_size
                
                # Ограничиваем, чтобы не вылезали за границы картинки
                x_min = max(0, x_min)
                y_min = max(0, y_min)
                x_max = min(new_w, x_max)
                y_max = min(new_h, y_max)
                
                x_center = ((x_min + x_max) / 2.0) / new_w
                y_center = ((y_min + y_max) / 2.0) / new_h
                width = (x_max - x_min) / new_w
                height = (y_max - y_min) / new_h
                
                class_id = CLASS_MAPPING[class_name]
                yolo_lines.append(f"{class_id} {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}")

    # --- 4.4. СОХРАНЯЕМ .txt ФАЙЛ ---
    txt_filename = Path(img_filename).stem + ".txt"
    dest_txt_path = os.path.join(OUTPUT_ROOT, "labels", split_type, txt_filename)
    
    with open(dest_txt_path, "w", encoding="utf-8") as f:
        f.write("\n".join(yolo_lines))

# ==========================================
# 5. ЗАПУСК ОБРАБОТКИ
# ==========================================
print("🔄 Конвертация Train набора...")
for jf in train_files: 
    convert_to_yolo(jf, "train")

print("🔄 Конвертация Val набора...")
for jf in val_files: 
    convert_to_yolo(jf, "val")

print("✅ Конвертация изображений и аннотаций завершена!")

# ==========================================
# 6. ГЕНЕРАЦИЯ data.yaml
# ==========================================
# Используем абсолютный путь для надежности при запуске из Jupyter/терминала
abs_output_root = os.path.abspath(OUTPUT_ROOT)

yaml_content = f"""path: {abs_output_root}
train: images/train
val: images/val

nc: 11
names:
  0: pawn
  1: rook
  2: knight
  3: bishop
  4: queen
  5: king
  6: chessboard
  7: corner_white_left
  8: corner_white_right
  9: corner_black_left
  10: corner_black_right

# Аугментации (оптимизированы для шахмат)
hsv_h: 0.015
hsv_s: 0.7
hsv_v: 0.4
degrees: 10.0
translate: 0.1
scale: 0.5
perspective: 0.005
flipud: 0.0
fliplr: 0.5
mosaic: 0.5
mixup: 0.0
"""

yaml_path = os.path.join(OUTPUT_ROOT, "data.yaml")
with open(yaml_path, "w", encoding="utf-8") as f:
    f.write(yaml_content)

print(f"📄 Файл конфигурации создан: {yaml_path}")
print("🚀 Датасет полностью готов к обучению!")

✅ Найдено 10000 аннотаций.
📊 Train: 8000 | Val: 2000
🔄 Конвертация Train набора...
🔄 Конвертация Val набора...
✅ Конвертация изображений и аннотаций завершена!
📄 Файл конфигурации создан: ./yolo_chess_dataset_final/data.yaml
🚀 Датасет полностью готов к обучению!


In [ ]:
import cv2
import os
import random
import numpy as np
from pathlib import Path


OUTPUT_ROOT = "/home/fedosdan2/Study/EdgeChess-AI/CV/yolo_chess_dataset_final"
IMAGE_TRAIN = f"{OUTPUT_ROOT}/images/train"
LABEL_TRAIN = f"{OUTPUT_ROOT}/labels/train"

CLASS_NAMES = {
    0: "pawn",
    1: "rook",
    2: "knight",
    3: "bishop",
    4: "queen",
    5: "king",
    6: "chessboard",
    7: "corner_white_left",
    8: "corner_white_right",
    9: "corner_black_left",
    10: "corner_black_right"
}

COLORS = {
    0: (0, 255, 0),
    1: (255, 0, 0),
    2: (0, 165, 255),
    3: (255, 255, 0),
    4: (255, 0, 255),
    5: (0, 0, 255),
    6: (255, 255, 255),
    7: (255, 255, 0),    # corner_white_left - желтый
    8: (255, 128, 0),    # corner_white_right - оранжевый
    9: (128, 0, 255),    # corner_black_left - фиолетовый
    10: (0, 255, 128)    # corner_black_right - бирюзовый
}


def draw_yolo_bbox(image, label_file):
    """Читает YOLO .txt файл и рисует bounding box'ы на изображении."""
    img_height, img_width = image.shape[:2]
    corners = {}

    if not os.path.exists(label_file):
        print(f"⚠️ Файл разметки не найден: {label_file}")
        return image, None

    with open(label_file, 'r', encoding='utf-8') as f:
        lines = f.readlines()

        for line in lines:
            parts = line.strip().split()
            if len(parts) != 5:
                continue

            class_id = int(parts[0])
            x_center = float(parts[1])
            y_center = float(parts[2])
            width = float(parts[3])
            height = float(parts[4])

            x_center_px = x_center * img_width
            y_center_px = y_center * img_height
            w_px = width * img_width
            h_px = height * img_height

            x_min = int(x_center_px - w_px / 2)
            y_min = int(y_center_px - h_px / 2)
            x_max = int(x_center_px + w_px / 2)
            y_max = int(y_center_px + h_px / 2)

            x_min = max(0, x_min)
            y_min = max(0, y_min)
            x_max = min(img_width, x_max)
            y_max = min(img_height, y_max)

            if class_id in [7, 8, 9, 10]:
                corners[class_id] = (int(x_center_px), int(y_center_px))
                
                # --- ИСПРАВЛЕНО: теперь это ВНУТРИ цикла ---
                cx = int(x_center_px)
                cy = int((y_center_px + h_px / 2) - (h_px / 10)) 
                cv2.drawMarker(image, (cx, cy), color, cv2.MARKER_CROSS, 20, 2)

            thickness = 3 if class_id == 6 else 2
            color = COLORS.get(class_id, (128, 128, 128))
            cv2.rectangle(image, (x_min, y_min), (x_max, y_max), color, thickness)

            label = CLASS_NAMES.get(class_id, f"class_{class_id}")
            (text_width, text_height), baseline = cv2.getTextSize(
                label, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 2
            )
            cv2.rectangle(image, (x_min, y_min - text_height - baseline), (x_min + text_width, y_min), color, -1)
            cv2.putText(image, label, (x_min, y_min - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0) if class_id == 6 else (255, 255, 255), 2)

        return image, corners

def get_ordered_corners_with_orientation(corners_dict, img_width, img_height):
    """
    Определяет порядок углов для Perspective Transform с правильной ориентацией.
    
    corners_dict: {class_id: (x, y)}
    - 7: corner_white_left
    - 8: corner_white_right
    - 9: corner_black_left
    - 10: corner_black_right
    
    Возвращает: (ordered_points, white_side_at_bottom)
    """
    # Извлекаем координаты углов
    white_left = np.array(corners_dict[7])
    white_right = np.array(corners_dict[8])
    black_left = np.array(corners_dict[9])
    black_right = np.array(corners_dict[10])
    
    # Определяем, где белая сторона (выше или ниже центра)
    white_center_y = (white_left[1] + white_right[1]) / 2
    black_center_y = (black_left[1] + black_right[1]) / 2
    
    white_at_bottom = white_center_y > black_center_y
    
    # Сортируем углы для Perspective Transform
    # Порядок: [top_left, top_right, bottom_right, bottom_left]
    if white_at_bottom:
        # Белая сторона внизу
        # Верхние углы = черные
        top_left = black_left if black_left[0] < black_right[0] else black_right
        top_right = black_right if black_left[0] < black_right[0] else black_left
        # Нижние углы = белые
        bottom_left = white_left if white_left[0] < white_right[0] else white_right
        bottom_right = white_right if white_left[0] < white_right[0] else white_left
    else:
        # Белая сторона вверху
        # Верхние углы = белые
        top_left = white_left if white_left[0] < white_right[0] else white_right
        top_right = white_right if white_left[0] < white_right[0] else white_left
        # Нижние углы = черные
        bottom_left = black_left if black_left[0] < black_right[0] else black_right
        bottom_right = black_right if black_left[0] < black_right[0] else black_left
    
    ordered = [top_left, top_right, bottom_right, bottom_left]
    
    return ordered, white_at_bottom


def apply_perspective_transform(image, corners, output_size=900, padding=50, white_at_bottom=True):
    """
    Применяет Perspective Transform по 4 углам доски с правильной ориентацией.
    """
    src_points = np.float32(corners)
    
    total_size = output_size + 2 * padding
    
    if white_at_bottom:
        # Стандартная ориентация: белая сторона (ряд 1) внизу
        dst_points = np.float32([
            [padding, padding],                      # top_left = a8
            [output_size + padding, padding],        # top_right = h8
            [output_size + padding, output_size + padding],  # bottom_right = h1
            [padding, output_size + padding]         # bottom_left = a1
        ])
    else:
        # Перевернутая ориентация: белая сторона вверху
        dst_points = np.float32([
            [padding, padding],                      # top_left = a1
            [output_size + padding, padding],        # top_right = h1
            [output_size + padding, output_size + padding],  # bottom_right = h8
            [padding, output_size + padding]         # bottom_left = a8
        ])
    
    M = cv2.getPerspectiveTransform(src_points, dst_points)
    warped = cv2.warpPerspective(image, M, (total_size, total_size))
    
    return warped, M


def draw_8x8_grid(image, output_size=900, padding=50, color=(0, 255, 255), thickness=1, white_at_bottom=True):
    """
    Рисует сетку 8x8 на изображении с учетом ориентации.
    """
    h, w = image.shape[:2]
    cell_size = output_size / 8
    
    # Вертикальные линии
    for i in range(1, 8):
        x = int(padding + i * cell_size)
        cv2.line(image, (x, padding), (x, padding + output_size), color, thickness)
    
    # Горизонтальные линии
    for i in range(1, 8):
        y = int(padding + i * cell_size)
        cv2.line(image, (padding, y), (padding + output_size, y), color, thickness)
    
    # Подписи колонок (a-h) и рядов (1-8)
    cols = "abcdefgh"
    for i in range(8):
        if white_at_bottom:
            # Буквы колонок снизу (ряд 1)
            cv2.putText(
                image, cols[i],
                (int(padding + i * cell_size + cell_size / 2 - 5), padding + output_size + 20),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1
            )
            # Цифры рядов слева (1-8 снизу вверх)
            cv2.putText(
                image, str(i + 1),
                (padding - 25, int(padding + (7 - i) * cell_size + cell_size / 2 + 5)),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1
            )
        else:
            # Перевернутая ориентация: буквы сверху, ряды 8-1 сверху вниз
            cv2.putText(
                image, cols[len(cols) - i - 1],
                (int(padding + i * cell_size + cell_size / 2 - 5), padding - 10),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1
            )
            cv2.putText(
                image, str(i+1),
                (padding - 25, int(padding + i * cell_size + cell_size / 2 + 5)),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1
            )
    
    return image


# ==========================================
# ЗАПУСК
# ==========================================
img_files = [f for f in os.listdir(IMAGE_TRAIN) if f.endswith(('.jpg', '.jpeg', '.png'))]

if not img_files:
    print("❌ Изображения в папке train не найдены! Проверь пути.")
else:
    img_name_no_ext = Path("/home/fedosdan2/Study/EdgeChess-AI/CV/yolo_chess_dataset_final/images/train/rgb_10.jpeg").stem
    label_path = "/home/fedosdan2/Study/EdgeChess-AI/CV/yolo_chess_dataset_final/labels/train/rgb_10.txt"
     
    selected_img = Path("/home/fedosdan2/Study/EdgeChess-AI/CV/yolo_chess_dataset_final/images/train/rgb_10.jpeg")
    img_path = "/home/fedosdan2/Study/EdgeChess-AI/CV/yolo_chess_dataset_final/images/train/rgb_10.jpeg"

   

    print(f"🖼️  Визуализация изображения: {selected_img}")
    print(f"📄  Файл разметки: {img_name_no_ext}.txt")

    img = cv2.imread(img_path)
    if img is None:
        print("❌ Не удалось прочитать изображение. Проверь путь.")
    else:
        img_height, img_width = img.shape[:2]

        img_annotated, corners = draw_yolo_bbox(img.copy(), label_path)

        required_corners = {7, 8, 9, 10}
        found_corners = set(corners.keys())
        missing_corners = required_corners - found_corners

        if missing_corners:
            print(f"⚠️ Не найдены углы: {[CLASS_NAMES[c] for c in missing_corners]}")
            output_path = f"visualized_{selected_img}"
            cv2.imwrite(output_path, img_annotated)
            print(f"✅ Сохранено: {output_path}")
        else:
            print(f"✅ Все 4 угла найдены:")
            for cid, (x, y) in corners.items():
                print(f"   {CLASS_NAMES[cid]}: ({x}, {y})")

            # Определяем порядок углов и ориентацию
            ordered_corners, white_at_bottom = get_ordered_corners_with_orientation(
                corners, img_width, img_height
            )
            
            print(f" Ориентация: белая сторона {'внизу' if white_at_bottom else 'вверху'}")

            # Perspective Transform
            output_size = 900
            padding = 50
            warped_board, M = apply_perspective_transform(
                img, ordered_corners, output_size, padding, white_at_bottom
            )

            # Рисуем сетку
            warped_with_grid = draw_8x8_grid(
                warped_board.copy(), output_size, padding, 
                color=(0, 255, 255), thickness=1, white_at_bottom=white_at_bottom
            )

            # Переносим фигуры
            if os.path.exists(label_path):
                with open(label_path, 'r', encoding='utf-8') as f:
                    lines = f.readlines()

                for line in lines:
                    parts = line.strip().split()
                    if len(parts) != 5:
                        continue

                    class_id = int(parts[0])
                    if class_id in [6, 7, 8, 9, 10]:
                        continue

                    x_center = float(parts[1]) * img_width
                    y_center = float(parts[2]) * img_height
                    height = float(parts[4]) * img_height  # <-- ДОБАВЛЕНО: читаем высоту bbox

                    # --- ГЛАВНОЕ ИСПРАВЛЕНИЕ ---
                    # Сдвигаем точку от центра вниз на 40% высоты bounding box.
                    # Это помещает точку ближе к основанию фигуры, компенсируя перспективу.
                    piece_x = x_center
                    piece_y = y_center + (height * 0.4)

                    # Трансформируем именно эту смещенную точку
                    point = np.float32([[[piece_x, piece_y]]])
                    transformed_point = cv2.perspectiveTransform(point, M)
                    tx, ty = transformed_point[0][0]

                    cv2.circle(warped_with_grid, (int(tx), int(ty)), 5,
                               COLORS.get(class_id, (128, 128, 128)), -1)

                    # Определяем клетку
                    col = int((tx - padding) / (output_size / 8))
                    row = int((ty - padding) / (output_size / 8))
                    col = max(0, min(7, col))
                    row = max(0, min(7, row))

                    if white_at_bottom:
                        cell_name = f"{'abcdefgh'[col]}{8 - row}"
                    else:
                        cell_name = f"{'hgfedcba'[col]}{row + 1}"
                    
                    label = CLASS_NAMES.get(class_id, "?")

                    cv2.putText(
                        warped_with_grid, f"{label}({cell_name})",
                        (int(tx) + 8, int(ty) + 4),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.35,
                        COLORS.get(class_id, (128, 128, 128)), 1
                    )

            # Собираем коллаж
            scale = 800 / img_height
            img_annotated_resized = cv2.resize(
                img_annotated,
                (int(img_width * scale), 800),
                interpolation=cv2.INTER_AREA
            )

            gap = 40
            collage_w = img_annotated_resized.shape[1] + gap + warped_board.shape[1]
            collage_h = max(800, warped_board.shape[0]) + 60

            collage = np.ones((collage_h, collage_w, 3), dtype=np.uint8) * 40

            cv2.putText(collage, "Original + BBox + Corners",
                        (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)
            cv2.putText(collage, f"Perspective Transform (white {'bottom' if white_at_bottom else 'top'})",
                        (img_annotated_resized.shape[1] + gap + 10, 30),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)

            offset_x = img_annotated_resized.shape[1] + gap
            collage[60:60 + img_annotated_resized.shape[0], 0:img_annotated_resized.shape[1]] = img_annotated_resized
            collage[60:60 + warped_board.shape[0], offset_x:offset_x + warped_board.shape[1]] = warped_with_grid

            output_path = f"collage_corners.jpeg"
            cv2.imwrite(output_path, collage)
            print(f"✅ Коллаж сохранен: {output_path}")

            warped_path = f"warped_corners.jpeg"
            cv2.imwrite(warped_path, warped_with_grid)
            print(f"✅ Выпрямленная доска сохранена: {warped_path}")

🖼️  Визуализация изображения: /home/fedosdan2/Study/EdgeChess-AI/CV/yolo_chess_dataset_final/images/train/rgb_10.jpeg
📄  Файл разметки: rgb_10.txt
✅ Все 4 угла найдены:
   corner_white_left: (803, 619)
   corner_white_right: (679, 294)
   corner_black_left: (287, 317)
   corner_black_right: (231, 661)
 Ориентация: белая сторона вверху
✅ Коллаж сохранен: collage_corners.jpeg
✅ Выпрямленная доска сохранена: warped_corners.jpeg
